# Gold Layer Data Health Checks
Validate row counts, schema consistency, join integrity, and dropped rows between Silver → Gold transformations.

In [ ]:
%%sql -r ctx
USE SCHEMA LEAGUE_RECORDS.GOLD;

## 1. Expected Row Relationships
Validate that gold grain matches expectations:
- `PLAYER_STATS_SUMMARY` should have 1 row per (MATCH_ID, PARTICIPANT_POS_ID) = same distinct combos as PLAYERS_SUMMARY_SILVER
- `MATCH_TEAM_STATS_SUMMARY` should have 1 row per MATCH_ID = same count as MATCHES_SUMMARY_SILVER
- `CHAMPION_OVERVIEW` row count ≈ distinct champions in CHAMPIONS_REF_SILVER (minus ID=0)

In [ ]:
%%sql -r grain_checks
WITH CHECKS AS (
    SELECT
        'PLAYER_STATS_SUMMARY' AS CHECK_NAME,
        (SELECT COUNT(*) FROM GOLD.PLAYER_STATS_SUMMARY) AS GOLD_ROWS,
        (SELECT COUNT(DISTINCT MATCH_ID || '|' || PARTICIPANT_POS_ID) FROM SILVER.PLAYERS_SUMMARY_SILVER) AS EXPECTED_ROWS
    UNION ALL
    SELECT
        'MATCH_TEAM_STATS_SUMMARY',
        (SELECT COUNT(*) FROM GOLD.MATCH_TEAM_STATS_SUMMARY),
        (SELECT COUNT(DISTINCT MATCH_ID) FROM SILVER.TEAM_INTERVAL_SILVER)
    UNION ALL
    SELECT
        'CHAMPION_OVERVIEW',
        (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEW),
        (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF_SILVER WHERE CHAMPION_ID != 0)
)

SELECT
    CHECK_NAME,
    GOLD_ROWS,
    EXPECTED_ROWS,
    GOLD_ROWS - EXPECTED_ROWS AS ROW_DIFF,
    CASE
        WHEN GOLD_ROWS = EXPECTED_ROWS THEN '✓ PASS'
        WHEN GOLD_ROWS < EXPECTED_ROWS THEN '✗ ROWS DROPPED'
        ELSE '⚠ MORE ROWS THAN EXPECTED'
    END AS STATUS
FROM CHECKS;

## 3. Join Integrity: Rows Dropped During Joins
Identify silver records that failed to join into gold (orphaned keys).

In [ ]:
%%sql -r player_join_drops
-- PLAYER_STATS_SUMMARY: players in silver that are missing from gold
-- This table joins PLAYER_INTERVAL_SILVER, PLAYERS_SUMMARY_SILVER, MATCHES_SUMMARY_SILVER
WITH silver_keys AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM SILVER.PLAYERS_SUMMARY_SILVER
),
gold_keys AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.PLAYER_STATS_SUMMARY
),
missing AS (
    SELECT s.MATCH_ID, s.PARTICIPANT_POS_ID
    FROM silver_keys s
    LEFT JOIN gold_keys g
        ON s.MATCH_ID = g.MATCH_ID AND s.PARTICIPANT_POS_ID = g.PARTICIPANT_POS_ID
    WHERE g.MATCH_ID IS NULL
)
SELECT
    COUNT(*) AS players_dropped,
    (SELECT COUNT(*) FROM silver_keys) AS total_silver_players,
    ROUND(COUNT(*) / (SELECT COUNT(*) FROM silver_keys) * 100, 2) AS pct_dropped
FROM missing;

In [ ]:
%%sql -r match_join_drops
-- MATCH_TEAM_STATS_SUMMARY: matches in silver that are missing from gold
WITH silver_matches AS (
    SELECT DISTINCT MATCH_ID FROM SILVER.MATCHES_SUMMARY_SILVER
),
gold_matches AS (
    SELECT DISTINCT MATCH_ID FROM GOLD.MATCH_TEAM_STATS_SUMMARY
),
missing AS (
    SELECT s.MATCH_ID
    FROM silver_matches s
    LEFT JOIN gold_matches g ON s.MATCH_ID = g.MATCH_ID
    WHERE g.MATCH_ID IS NULL
)
SELECT
    COUNT(*) AS matches_dropped,
    (SELECT COUNT(*) FROM silver_matches) AS total_silver_matches,
    ROUND(COUNT(*) / (SELECT COUNT(*) FROM silver_matches) * 100, 2) AS pct_dropped
FROM missing;

In [ ]:
%%sql -r drop_root_causes
-- Why are rows dropped? Diagnose root cause for PLAYER_STATS_SUMMARY
-- Players exist in PLAYERS_SUMMARY_SILVER but have no interval data
WITH players_without_intervals AS (
    SELECT PS.MATCH_ID, PS.PARTICIPANT_POS_ID
    FROM SILVER.PLAYERS_SUMMARY_SILVER PS
    LEFT JOIN SILVER.PLAYER_INTERVAL_SILVER PIV
        ON PIV.MATCH_ID = PS.MATCH_ID
        AND PIV.PARTICIPANT_POS_ID = PS.PARTICIPANT_POS_ID
    WHERE PIV.MATCH_ID IS NULL
)
SELECT
    'Players with no interval data' AS drop_reason,
    COUNT(*) AS affected_rows
FROM players_without_intervals
UNION ALL
-- Matches in PLAYER_INTERVAL but not in MATCHES_SUMMARY (broken FK)
SELECT
    'Intervals with no match summary' AS drop_reason,
    COUNT(DISTINCT PIV.MATCH_ID)
FROM SILVER.PLAYER_INTERVAL_SILVER PIV
LEFT JOIN SILVER.MATCHES_SUMMARY_SILVER MAT
    ON MAT.MATCH_ID = PIV.MATCH_ID
WHERE MAT.MATCH_ID IS NULL;

In [ ]:
%%sql -r null_audit
SELECT 'PLAYER_STATS_SUMMARY' AS gold_table, 'MATCH_ID' AS column_name,
    SUM(CASE WHEN MATCH_ID IS NULL THEN 1 ELSE 0 END) AS null_count,
    COUNT(*) AS total_rows
FROM GOLD.PLAYER_STATS_SUMMARY
UNION ALL
SELECT 'PLAYER_STATS_SUMMARY', 'PARTICIPANT_POS_ID',
    SUM(CASE WHEN PARTICIPANT_POS_ID IS NULL THEN 1 ELSE 0 END), COUNT(*)
FROM GOLD.PLAYER_STATS_SUMMARY
UNION ALL
SELECT 'PLAYER_STATS_SUMMARY', 'CHAMPION',
    SUM(CASE WHEN CHAMPION IS NULL THEN 1 ELSE 0 END), COUNT(*)
FROM GOLD.PLAYER_STATS_SUMMARY
UNION ALL
SELECT 'MATCH_TEAM_STATS_SUMMARY', 'MATCH_ID',
    SUM(CASE WHEN MATCH_ID IS NULL THEN 1 ELSE 0 END), COUNT(*)
FROM GOLD.MATCH_TEAM_STATS_SUMMARY
UNION ALL
SELECT 'MATCH_TEAM_STATS_SUMMARY', 'WINNING_TEAM',
    SUM(CASE WHEN WINNING_TEAM IS NULL THEN 1 ELSE 0 END), COUNT(*)
FROM GOLD.MATCH_TEAM_STATS_SUMMARY
UNION ALL
SELECT 'CHAMPION_OVERVIEW', 'CHAMPION_NAME',
    SUM(CASE WHEN CHAMPION_NAME IS NULL THEN 1 ELSE 0 END), COUNT(*)
FROM GOLD.CHAMPION_OVERVIEW;

## 5. Spot-Check: Sample Records Gold vs Silver
Pull a few records and compare key metrics to verify transformation correctness.

In [ ]:
WITH sample_player AS (
    SELECT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.PLAYER_STATS_SUMMARY
    LIMIT 3
),
silver_last_interval AS (
    SELECT *
    FROM SILVER.PLAYER_INTERVAL_SILVER
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MATCH_ID, PARTICIPANT_POS_ID
        ORDER BY MINUTE DESC
    ) = 1
)
SELECT
    g.MATCH_ID,
    g.PARTICIPANT_POS_ID,
    g.KILLS AS gold_kills,
    g.DEATHS AS gold_deaths,
    g.ASSISTS AS gold_assists,
    g.TOTAL_GOLD AS gold_total_gold,
    piv.KILLS AS silver_last_kills,
    piv.DEATHS AS silver_last_deaths,
    piv.ASSISTS AS silver_last_assists,
    piv.TOTAL_GOLD AS silver_last_gold,
    piv.MINUTE AS silver_last_minute,
    -- Verify joined from Gold is same as silver
    CASE 
        WHEN g.KILLS = piv.KILLS 
            AND g.DEATHS = piv.DEATHS
            AND g.ASSISTS = piv.ASSISTS 
            AND g.TOTAL_GOLD = piv.TOTAL_GOLD
         THEN '✓ MATCH'
        ELSE '✗ MISMATCH' 
    END AS verification
FROM GOLD.PLAYER_STATS_SUMMARY g
JOIN sample_player sp
    ON g.MATCH_ID = sp.MATCH_ID AND g.PARTICIPANT_POS_ID = sp.PARTICIPANT_POS_ID
JOIN silver_last_interval piv
    ON piv.MATCH_ID = g.MATCH_ID
    AND piv.PARTICIPANT_POS_ID = g.PARTICIPANT_POS_ID;

## 6. Duplicate Detection
Gold tables should have no duplicate keys at their expected grain.

In [ ]:
%%sql -r duplicate_check
-- Check for duplicate keys in gold tables
SELECT 'PLAYER_STATS_SUMMARY' AS gold_table, 
    'MATCH_ID + PARTICIPANT_POS_ID' AS grain,
    COUNT(*) - COUNT(DISTINCT MATCH_ID || '|' || PARTICIPANT_POS_ID) AS duplicate_rows
FROM GOLD.PLAYER_STATS_SUMMARY
UNION ALL
SELECT 'MATCH_TEAM_STATS_SUMMARY',
    'MATCH_ID',
    COUNT(*) - COUNT(DISTINCT MATCH_ID)
FROM GOLD.MATCH_TEAM_STATS_SUMMARY
UNION ALL
SELECT 'CHAMPION_OVERVIEW',
    'CHAMPION_ID',
    COUNT(*) - COUNT(DISTINCT CHAMPION_ID)
FROM GOLD.CHAMPION_OVERVIEW
UNION ALL
SELECT 'CHAMPION_INTERVALS',
    'CHAMPION + MINUTE',
    COUNT(*) - COUNT(DISTINCT CHAMPION || '|' || MINUTE)
FROM GOLD.CHAMPION_INTERVALS;

## 7. Summary Health Score

In [ ]:
# Aggregate health results into a summary score
import pandas as pd

checks = []

# Grain checks
grain_df = grain_checks.to_pandas()
for _, row in grain_df.iterrows():
    checks.append({
        'check': row['CHECK_NAME'],
        'passed': str(row['STATUS']).startswith('✓')
    })

# Duplicate checks
dup_df = duplicate_check.to_pandas()
for _, row in dup_df.iterrows():
    checks.append({
        'check': f"No duplicates in {row['GOLD_TABLE']}",
        'passed': int(row['DUPLICATE_ROWS']) == 0
    })

# NULL audit
null_df = null_audit.to_pandas()
for _, row in null_df.iterrows():
    checks.append({
        'check': f"No NULLs in {row['GOLD_TABLE']}.{row['COLUMN_NAME']}",
        'passed': int(row['NULL_COUNT']) == 0
    })

df = pd.DataFrame(checks)
passed = df['passed'].sum()
total = len(df)

print(f"{'='*50}")
print(f"  GOLD LAYER HEALTH SCORE: {passed}/{total} checks passed")
print(f"{'='*50}")
print()
for _, row in df.iterrows():
    icon = '✓' if row['passed'] else '✗'
    print(f"  {icon}  {row['check']}")

if passed < total:
    print(f"\n  ⚠ {total - passed} issue(s) require investigation.")
else:
    print(f"\n  All checks passed. Gold layer is consistent with silver.")